In [35]:
import numpy as np

In [56]:

def expected_new_places(state, action, layout, circle):

    def rolls_security_dice():
        possible_trap_triggered = [False]
        possible_dice_rolls = [0, 1]
        p = 1/2
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    def rolls_normal_dice():
        possible_trap_triggered = [True, False]
        possible_dice_rolls = [0, 1, 2]
        p = 1/6
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    


    def rolls_risky_dice():
        possible_trap_triggered = [True]
        possible_dice_rolls = [0, 1, 2, 3]
        p = 1/4
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    rolls_dice_functions = {0:rolls_security_dice, 1:rolls_normal_dice, 2:rolls_risky_dice} 



    (current_position, current_skip_next_turn) = state

    
    if current_skip_next_turn:
        new_state = (current_position, False)
        return [(1, new_state)]
    

    # roll dices
    roll_function = rolls_dice_functions[action]
    list_rolls =  roll_function()

    list_new_positions_before_traps = []

    for (p, trap_triggered, dice_roll) in list_rolls:
        # find new position
        if dice_roll==0:
            new_position_before_trap = current_position
            list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))
        else: # dice roll is not 0
            if (current_position == 2):
                    new_position_before_trap = current_position + dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))
                    new_position_before_trap = 9+dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))


            elif  current_position in range(10): # but not 2
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 10:
                    if circle:
                        new_position_before_trap -= 10
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

            elif current_position in range(10, 14):
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 14:
                    if circle:
                        new_position_before_trap -=14
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

    list_new_places_after_traps = []

    for (p, new_position_before_trap, trap_triggered) in list_new_positions_before_traps:
        # Deal with the traps
        trap = layout[new_position_before_trap]

        if  (not trap_triggered) or (trap == 0):
            new_skip_next_turn = False
            new_position_after_trap = new_position_before_trap
            list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))
        else:
            trap_list = []

            if trap == 4:
                trap_list.append(1)
                trap_list.append(2)
                trap_list.append(3)
                p/=3
            else:
                trap_list.append(trap)
            
            for trap in trap_list:
                if   trap == 1:
                    new_position_after_trap = 0
                    new_skip_next_turn = False

                elif trap == 2:
                    new_position_after_trap = new_position_before_trap
                    if new_position_after_trap in range(10, 13):
                        new_position_after_trap -= 7 # -7 -3 = -10
                    new_position_after_trap = max(0, new_position_after_trap - 3)
                    new_skip_next_turn = False


                elif trap == 3:
                    new_position_after_trap = new_position_before_trap
                    new_skip_next_turn=True

                list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))

    return list_new_places_after_traps




def expected_probability_win(state, action, P, layout, circle):
    # state = (position1:(), positions2:(), whoseturn)
    # probabilité de gagner du joueur 1
    # if whoseturn==1:-> max
    # if whoseturn==2:-> min

    (my_place, other_place, whoseturn) = state

    if other_place[0] == 14:
        if whoseturn==2: return 1
        else: return 0
    
    list_expected_my_new_places = expected_new_places(my_place, action, layout, circle)

    new_probability  = 0
    for (p, new_place) in list_expected_my_new_places:
        new_probability += p*P[(other_place, new_place, (whoseturn-1)*1 + (2-whoseturn)*2)]

    return new_probability

    
    
def min_max(layout, circle, theta):

    # probabilité de gagner du joueur 1
    # if whoseturn==1:-> max
    # if whoseturn==2:-> min
    
    possible_places = []
    for i in range(14):
        if layout[i]>=3:
            possible_places.append((i, False))
            possible_places.append((i, True))
        else:
            possible_places.append((i, False))

    possible_states = []
    for place1 in possible_places:
        for place2 in possible_places:
            possible_states.append((place1, place2, 1))
            possible_states.append((place1, place2, 2))

        possible_states.append((place1, (14, False), 1))
        possible_states.append((place1, (14, False),  2))


    P = {}
    best_policy = {}

    for state in possible_states:
        P[state] = 0
        best_policy[state] = 0
    
    delta = 2*theta
    while delta>=theta:
        print(delta)
        delta = 0
        for state in possible_states:
            v = P[state]
            (_, _, whoseturn) = state

            if whoseturn == 2:
                minv = np.inf
                for action in range(3):
                    Pn = expected_probability_win(state, action, P, layout, circle)
                    # print(nv)
                    if Pn <= minv:
                        minv = Pn
                        best_policy[state] = action
                P[state] = minv
            else :  # whoseturn == 1
                maxv = -np.inf
                for action in range(3):
                    Pn = expected_probability_win(state, action, P, layout, circle)
                    if Pn >= maxv:
                        maxv = Pn
                        best_policy[state] = action
                P[state] = maxv
            delta = max(abs(v-P[state]), delta)
    print(delta)
    return P, best_policy
        


In [57]:
# circle: a boolean variable (type bool), indicating if the player must land exactly on
# the final, goal, square 15 to win (circle = True) or still wins by overstepping the final
# square (circle = False).
circle = True

# layout: a vector of type numpy.ndarray that represents the layout of the game, containing 15 values
#         representing the 15 squares of the Snakes and Ladders game:
# layout[i] = 0 if it is an ordinary square
#           = 1 if it is a “restart” trap (go back to square 1)
#           = 2 if it is a “penalty” trap (go back 3 steps)
#           = 3 if it is a “prison” trap (skip next turn)
#           = 4 if it is a “mystery” trap (random effect among the three previous)
# Note that the first and final squares cannot be trapped.

layout = np.ones(15)*4
layout[0] = 0
layout[14] = 0

min_max(layout, circle, 0.00001)

# 1512


2e-05
1
0.5
0.5
0.25
0.13835573131001366
0.13073850060404013
0.12074524996929342
0.10725009614308006
0.10445500824056497
0.0960763451574344
0.09049619421536259
0.08310971616460877
0.07732722031580797
0.06940720987329185
0.06429541680671386
0.05609301873460476
0.05020697613365971
0.04433801645830093
0.03730661124863366
0.030431178782570623
0.024495487257910897
0.02035012526455504
0.016608106853316706
0.013364179978366475
0.010504896987735712
0.008223926967303008
0.006366678503374634
0.005049135365718438
0.004164795191027804
0.0034138142924096115
0.002781672685426839
0.0022539942511438094
0.0018169837208293904
0.0014577028964783967
0.0011643514486036066
0.0009264285238460568
0.0007345983234916531
0.0005808359436371147
0.0004582338972285438
0.0003609074053617656
0.00028391643149783263
0.00022316855125048551
0.00017532298544742364
0.00013768478368036963
0.000108099410579654
8.485586916395782e-05
6.66012337041666e-05
5.226842101380136e-05
4.1017077584748485e-05
3.218601137777721e-05
2.52515

({((0, False), (0, False), 1): 0.5175166770959315,
  ((0, False), (0, False), 2): 0.4824173908808499,
  ((0, False), (1, False), 1): 0.46497072023513947,
  ((0, False), (1, False), 2): 0.5349793931772902,
  ((0, False), (1, True), 1): 0.5005533265953352,
  ((0, False), (1, True), 2): 0.49939076110633984,
  ((0, False), (2, False), 1): 0.39451254008888287,
  ((0, False), (2, False), 2): 0.605453473210366,
  ((0, False), (2, True), 1): 0.42939041160084734,
  ((0, False), (2, True), 2): 0.5705715333863893,
  ((0, False), (3, False), 1): 0.5482112574016388,
  ((0, False), (3, False), 2): 0.45173724697221,
  ((0, False), (3, True), 1): 0.5832194335837073,
  ((0, False), (3, True), 2): 0.4167249618473008,
  ((0, False), (4, False), 1): 0.5316826003978796,
  ((0, False), (4, False), 2): 0.4682834094034715,
  ((0, False), (4, True), 1): 0.5682511342822916,
  ((0, False), (4, True), 2): 0.4317119877303523,
  ((0, False), (5, False), 1): 0.49490230765551374,
  ((0, False), (5, False), 2): 0.5050